<a href="https://colab.research.google.com/github/ryojifukagawa-pixel/heatbox_izena/blob/main/%E3%82%A4%E3%82%BC%E3%83%8A_%E4%BF%9D%E8%AD%B7%E7%86%B1%E7%AE%B1_intearctive_model_%E3%82%B9%E3%83%88%E3%83%AA%E3%83%BC%E3%83%A0%E3%83%A9%E3%82%A4%E3%83%B3%E3%81%A8%E5%B1%A4%E7%8A%B6%E6%85%8B%E8%A1%A8%E7%8F%BE%E3%81%82%E3%82%8A_Untitled14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [ ]:
!pip install ipywidgets
from ipywidgets import interact, FloatSlider, ToggleButtons
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 物性値（代表温度 22.5℃）
# -----------------------------
beta = 3.0e-4
nu   = 0.9e-6
alpha = 0.14e-6
k = 0.6
g = 9.81

# -----------------------------
# Rayleigh 数
# -----------------------------
def calc_Ra(H, dT):
    return g * beta * dT * H**3 / (nu * alpha)

# -----------------------------
# Nusselt 数
# -----------------------------
def calc_Nu(Ra, C):
    return C * Ra**(1/3)

# -----------------------------
# 熱貫流率
# -----------------------------
def calc_U(Nu, H):
    return Nu * k / H

# -----------------------------
# 可視化（簡易モデル）
# -----------------------------
def visualize_field(mode):
    nx, ny = 40, 40
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)

    plt.figure(figsize=(6,5))

    if mode == "Bottom heating (convection ON)":
        # 対流セルの簡易モデル
        A = 1.0
        u =  A * np.sin(np.pi * X) * np.cos(np.pi * Y)
        v = -A * np.cos(np.pi * X) * np.sin(np.pi * Y)

        speed = np.sqrt(u**2 + v**2)

        plt.streamplot(X, Y, u, v, color=speed, cmap='turbo')
        plt.title("Convection streamlines (Bottom heating)")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.colorbar(label="Velocity magnitude")

    else:
        # 上側加熱 → 対流なし → 等温線
        T = 1 - Y  # 線形温度分布
        plt.contourf(X, Y, T, levels=20, cmap='coolwarm')
        plt.title("Isotherms (Top heating)")
        plt.xlabel("x")
        plt.ylabel("y")
        plt.colorbar(label="Temperature")

    plt.gca().invert_yaxis()  # 上が冷却面になるように
    plt.show()

# -----------------------------
# インタラクティブ表示関数
# -----------------------------
def interactive_model(H_mm, dT, C, mode):
    H = H_mm / 1000

    if mode == "Bottom heating (convection ON)":
        Ra = calc_Ra(H, dT)
        Nu = calc_Nu(Ra, C)
    else:
        Ra = calc_Ra(H, dT)
        Nu = 1.0

    U = calc_U(Nu, H)

    print(f"Mode = {mode}")
    print(f"Rayleigh number Ra = {Ra:.3e}")
    print(f"Nusselt number Nu = {Nu:.2f}")
    print(f"U-value = {U:.1f} W/(m²·K)")

    # 厚さスイープ
    H_list = np.linspace(0.03, 0.12, 50)
    U_list = []

    for H_i in H_list:
        Ra_i = calc_Ra(H_i, dT)
        if mode == "Bottom heating (convection ON)":
            Nu_i = calc_Nu(Ra_i, C)
        else:
            Nu_i = 1.0
        U_i = calc_U(Nu_i, H_i)
        U_list.append(U_i)

    plt.figure(figsize=(7,5))
    plt.plot(H_list*1000, U_list, lw=2)
    plt.axvline(H_mm, color='red', linestyle='--', label='Current H')
    plt.xlabel("Thickness H [mm]")
    plt.ylabel("U-value [W/(m²·K)]")
    plt.title("Effect of water layer thickness on U-value")
    plt.grid(True)
    plt.legend()
    plt.show()

    # 温度場・流れ場の可視化
    visualize_field(mode)

# -----------------------------
# UI
# -----------------------------
interact(
    interactive_model,
    H_mm = FloatSlider(value=90, min=30, max=120, step=1, description="H [mm]"),
    dT   = FloatSlider(value=15, min=5, max=25, step=1, description="ΔT [K]"),
    C    = FloatSlider(value=0.069, min=0.03, max=0.12, step=0.001, description="C coeff"),
    mode = ToggleButtons(
        options=["Bottom heating (convection ON)", "Top heating (convection OFF)"],
        description="Heating mode"
    )
);


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 15.4 MB/s eta 0:00:00


interactive(children=(FloatSlider(value=90.0, description='H [mm]', max=120.0, min=30.0, step=1.0), FloatSlide…